In [25]:
# Небольшой комментарий к коду:
# изначально пойдут ячейки с импортом модулей, нужных для работы 
# затем все реализованные функции
# и наконец, демонстрация работы

# Рекомендую прочиать мой README.md на GitHub, там есть важные комментарии.

In [2]:
# Импорт всех необходимых модулей для работы
import json
from collections import defaultdict, Counter
import pprint

In [3]:
# Загрузка данных
def load_data(file_name: str) -> list[list[int]]:
    """
    Загружает данные из JSON-файла, где каждая строка является отдельным JSON-объектом.

    Функция построчно читает файл, парсит каждую непустую строку как JSON
    и добавляет полученный объект в список.

    Args
    ----------
        file_name (str): Путь к JSON-файлу, где каждая строка содержит JSON-массив целых чисел.

    Returns
    ----------
        list[list[int]]: Список сессий, где каждая сессия представлена списком целых чисел.
    """
    sessions = []
    with open(file_name) as file:
        for line in file:
            line = line.strip()
            if line:
                sessions.append(json.loads(line))
    return sessions

In [4]:
def train_test_split(sessions: list[list[int]]) -> tuple[list[list[int]], list[int]]:
    """
    Разбиение сессий на train и test.

    Для каждой сессии:
      - все товары кроме последнего становятся
        обучающей сессией
      - последний товар становится тестовой целью

    Возвращаемые списки выровнены по индексу:
    test_targets[i] — товар, который нужно
    предсказать по train_sessions[i] как истории.

    Args
    ----------
    sessions : list of lists of ints
        Каждый вложенный список — одна сессия ID
        товаров, упорядоченная по времени.
        Все сессии содержат не менее 3 товаров.

    Returns
    -------
    train_sessions : list of lists of ints
        Сессии для обучения (исходные без последнего товара).
    test_targets : list of ints
        Следующий товар для предсказания по каждой сессии.
    """
    train_sessions = [session[:-1] for session in sessions]
    test_targets = [session[-1] for session in sessions]
    return train_sessions, test_targets

In [7]:
def build_transition_graph(train_sessions: list[list[int]]) -> dict[int, dict[int, float]]:
    """
    Построение графа переходов с вероятностями P(j | i).

    Для каждой обучающей сессии считаются все переходы между
    соседними товарами из i в j. По собранным связям
    вычисляются условные вероятности:
        P(j | i) = (количество случаев i перешло в j) / (сумма всех переходов из i в любой товар, в том числе j)

    Вершины графа - ID товаров. Ребра направленные, каждое ребро из i в j имеет вес - P(j | i).

    Args
    ----------
    train_sessions
        Список обучающих сессий.

    Returns
    -------
    dict[int, dict[int, float]]
        Словарь, где ключ - это текущий товар i,
        значение - это словарь {j: P(j | i)}.
        Если из товара i нет переходов, он отсутствует
        в возвращаемом словаре. 
    """    
    # Создаю словарь с ключом - ID и словарем, который автоматически считает количество связей.
    transitions_btw_items = defaultdict(Counter)

    # Перебор всех сессий.
    for session in train_sessions:
        # Перебор всех предметов.
        for current_item, next_item in zip(session, session[1:]):
            transitions_btw_items[current_item][next_item] += 1
    
    graph_probabilities = dict()
    for current_item, next_items_counts in transitions_btw_items.items():
        total_transitions = sum(next_items_counts.values())
        graph_probabilities[current_item] = {j_item: transitions_count / total_transitions 
                                                for j_item, transitions_count in next_items_counts.items()}
    
    return graph_probabilities

In [9]:
def get_most_popular_items(train_sessions: list[list[int]], k: int = 10) -> list[int]:
    """
    Возвращает список k товаров, отсортированных по убыванию популярности.
    
    Args
    ----------
    train_sessions
        Список обучающих сессий
    k
        Парамет, отвечающмий за колчество возвращаемых предметов
        
    Returns
    -------
    list[int]
        Список товаров, отсортированных по частоте встречаемости.
    """
    # Собираю все товары из всех сессий
    all_items = []
    for session in train_sessions:
        all_items.extend(session)
    
    # Считаю частоту каждого товара
    popularity = Counter(all_items)
    
    # Сортирую по убыванию частоты
    sorted_items = [item for item, _ in popularity.most_common()]
    
    return sorted_items[:k]

In [17]:
def recommend_next(graph_probabilities: dict[int, dict[int, float]], 
                   last_item: int, most_popular_items: list[int], k: int = 10) -> list[int]:
    """
    Рекомендация следующих товаров на основе графа переходов.
    
    Если для last_item есть переходы - возвращает топ-k самых вероятных.
    Если нет — возвращаем топ-k популярных товаров.
    
    Args
    ----------
    graph_probabilities
        Граф вероятностей переходов из build_transition_graph
    last_item
        Последнего товар в сессии
    most_popular_items
        Список товаров, отсортированных по популярности
        
    Returns
    -------
    list[int]
        Список из k рекомендуемых товаров
    """
    recommendations = []
    
    # Проверяю - есть ли переходы из последнего товара?
    if last_item in graph_probabilities:
        # Сортирую по убыванию вероятности
        sorted_probabilities = sorted(
            graph_probabilities[last_item].items(),
            key=lambda x: -x[1]
        )
        
        recommendations = [item for item, _ in sorted_probabilities[:k]]
    
    # Если рекомендаций меньше k. Добавляю популярные товары
    if len(recommendations) < k:
        for item in most_popular_items:
            if item != last_item and item not in recommendations:
                recommendations.append(item)
            if len(recommendations) == k:
                break
    
    return recommendations[:k]

In [10]:
def hit_at_k(recommendations: list[list[int]], true_items: list[int], k: int = 10) -> float:
    """
    Вычисление Hit@K для списка предсказаний.

    Parameters
    ----------
    recommendations : list of lists of ints
        recommendations[i] — ранжированный список
        рекомендаций для i-го примера.
    true_items : list of ints
        true_items[i] — истинный следующий товар
        для i-го примера.
    k : int
        Отсечка top-K (по умолчанию 10).

    Returns
    -------
    float
        Hit@K, значение от 0 до 1.
    """
    assert len(recommendations) == len(true_items), \
        "recommendations и true_items должны совпадать по длине"

    hits = 0
    for recs, true_item in zip(recommendations, true_items):
        if true_item in recs[:k]:
            hits += 1

    return hits / len(true_items)

In [11]:
# Загрузка сессий
sessions = load_data("sessions.jsonl")

In [12]:
# Формирование обучающей и тестовой выборок.
train_sessions, test_targets = train_test_split(sessions)

In [14]:
# Формирую граф вероятностей согласно входным сессиям.
graph_probabilities = build_transition_graph(train_sessions)

In [15]:
# Получаю самые популярные товары на случай, если будет мало связей у конкретного товара.
most_popular_items = get_most_popular_items(train_sessions)

In [18]:
# Делаю предсказания для каждой сессии
predictions = []
for session in train_sessions:
    last_item = session[-1]
    recomendations = recommend_next(graph_probabilities, last_item, most_popular_items)
    predictions.append(recomendations)

In [19]:
# Вычисляю Hit@10 для моей модели.
hit_at_10 = hit_at_k(predictions, test_targets, k=10)

In [20]:
# Для сравнения - бейзлайн - только популярные товары.
baseline_recommendations = [most_popular_items[:10] for _ in range(len(test_targets))]
baseline_hit = hit_at_k(baseline_recommendations, test_targets, k=10)

In [24]:
print("Финальное сравнение результатов:\n")
print(f"Hit@10 моей модели: {hit_at_10:.5f}\n")
print(f"Бейзлайн - только популярные - Hit@10: {baseline_hit:.5f}\n")

# Разница между моей моделью и бейзлайном.
difference = hit_at_10 - baseline_hit 
print(f"Разница качества моделей - {difference}")

Финальное сравнение результатов:

Hit@10 моей модели: 0.51423

Бейзлайн - только популярные - Hit@10: 0.38402

Разница качества моделей - 0.13021442495126706
